# Ask User Failure Mode Analysis

This notebook analyzes failure modes in ask_user tool calls across multiple experiment runs.

**Tables Generated:**
1. % of trials with each **high-level** taxonomy failure mode
2. % of trials with each **low-level** taxonomy failure mode
3. % of trials **with a question** with each high-level taxonomy failure mode
4. % of trials **with a question** with each low-level taxonomy failure mode
5. % of trials **that failed** with each high-level taxonomy failure mode
6. % of trials **that failed** with each low-level taxonomy failure mode

In [45]:
import json
import sys
from pathlib import Path
from typing import Dict, List, Optional

import pandas as pd

# Add project root to path
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

from evaluation.ask_user_types import ALL_FLAGS, FLAG_CATEGORIES

# Configuration
PASS_THRESHOLD = 0.75

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.1%}'.format)

In [46]:
# Taxonomy structure (imported from ask_user_types)
print("High-Level Categories and their Low-Level Flags:")
print("=" * 60)
for category, flags in FLAG_CATEGORIES.items():
    print(f"\n{category}:")
    for flag in flags:
        print(f"  - {flag}")

High-Level Categories and their Low-Level Flags:

Question Quality:
  - flag_vague_questions
  - flag_compound_questions
  - flag_closed_questions

Question Targeting:
  - flag_missed_critical_segments
  - flag_wrong_dimension_focus
  - flag_inferable_information_asked

Information Integration:
  - flag_ignoring_user_response
  - flag_incorrect_information_application
  - flag_partial_information_utilization

Over-Clarification:
  - flag_redundant_questions
  - flag_unnecessary_questions
  - flag_excessive_cost

Under-Clarification:
  - flag_premature_execution
  - flag_insufficient_question_depth
  - flag_false_confidence

Timing & Strategy:
  - flag_late_clarification
  - flag_upfront_question_dump
  - flag_inefficient_question_sequence

User Response Misinterpretation:
  - flag_literal_misunderstanding
  - flag_context_misapplication
  - flag_implicit_information_missed


In [47]:
import re

def extract_short_run_label(run_name: str) -> str:
    """
    Extract a short, readable label from a run directory name.
    
    Examples:
        'run_sampled_underspecified_with_user_new_gpt-5.2_20260128_044134' -> 'gpt-5.2 (new)'
        'run_sampled_underspecified_with_user_claude-opus-4-5-20251101_20260124_023156' -> 'claude-opus-4-5'
        'run_sampled_underspecified_with_varied_user_new_claude-sonnet-4-5-20250929_20260128_044231' -> 'claude-sonnet-4-5 (varied)'
    """
    # Extract model name (pattern: last part before timestamp YYYYMMDD_HHMMSS)
    match = re.search(r'_([a-zA-Z][\w\-\.]+)_(\d{8}_\d{6})$', run_name)
    if match:
        model = match.group(1)
    else:
        model = run_name
    
    # Add suffix based on run type
    suffix_parts = []
    if '_new_' in run_name:
        suffix_parts.append('new')
    if '_varied_user_' in run_name:
        suffix_parts.append('varied')
    if '_family_' in run_name:
        suffix_parts.append('family')
    
    if suffix_parts:
        return f"{model} ({', '.join(suffix_parts)})"
    return model


def load_ask_user_data(run_dirs: List[Path]) -> pd.DataFrame:
    """
    Load and combine ask_user.csv files from multiple run directories.
    
    Args:
        run_dirs: List of paths to run directories (each must contain ask_user.csv)
        
    Returns:
        Combined DataFrame with all trials and a 'run_name' column
    """
    dfs = []
    
    for run_dir in run_dirs:
        run_path = Path(run_dir)
        csv_path = run_path / "ask_user.csv"
        
        if not csv_path.exists():
            print(f"Warning: ask_user.csv not found in {run_path}")
            continue
            
        df = pd.read_csv(csv_path)
        
        # Add run name (full directory name for uniqueness)
        df['run_name'] = run_path.name
        
        # Add short label for display
        df['run_label'] = extract_short_run_label(run_path.name)
        
        # Extract model name from run directory name
        model_name = df['model'].iloc[0] if 'model' in df.columns and len(df) > 0 else run_path.name
        df['run_model'] = model_name
        
        dfs.append(df)
        print(f"Loaded {len(df)} trials from {run_path.name}")
    
    if not dfs:
        raise ValueError("No ask_user.csv files found in provided directories")
    
    combined = pd.concat(dfs, ignore_index=True)
    print(f"\nTotal: {len(combined)} trials from {len(dfs)} runs")
    
    return combined


def add_derived_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Add derived columns for analysis."""
    df = df.copy()
    
    # Add passed/failed column
    df['passed'] = df['coverage_score'] >= PASS_THRESHOLD
    df['failed'] = df['coverage_score'] < PASS_THRESHOLD
    
    # Add has_question column
    df['has_question'] = df['ask_user_count'] > 0
    
    # Add high-level category flags (True if any flag in category is True)
    for category, flags in FLAG_CATEGORIES.items():
        category_col = f"category_{category.lower().replace(' ', '_').replace('&', 'and')}"
        available_flags = [f for f in flags if f in df.columns]
        if available_flags:
            df[category_col] = df[available_flags].any(axis=1)
        else:
            df[category_col] = False
    
    return df

In [48]:
def compute_failure_mode_stats(
    df: pd.DataFrame,
    group_by: Optional[str] = None,
    filter_col: Optional[str] = None,
    filter_value: Optional[bool] = None,
) -> Dict[str, pd.DataFrame]:
    """
    Compute failure mode statistics.
    
    Args:
        df: DataFrame with trial data
        group_by: Column to group by (e.g., 'run_name', 'run_model')
        filter_col: Column to filter on (e.g., 'has_question', 'failed')
        filter_value: Value to filter for
        
    Returns:
        Dict with 'high_level' and 'low_level' DataFrames
    """
    # Apply filter if specified
    if filter_col is not None and filter_value is not None:
        df_filtered = df[df[filter_col] == filter_value].copy()
    else:
        df_filtered = df.copy()
    
    if len(df_filtered) == 0:
        print(f"Warning: No trials match filter {filter_col}={filter_value}")
        return {'high_level': pd.DataFrame(), 'low_level': pd.DataFrame()}
    
    # Get category columns
    category_cols = [col for col in df_filtered.columns if col.startswith('category_')]
    
    # Get flag columns that exist in the data
    flag_cols = [f for f in ALL_FLAGS if f in df_filtered.columns]
    
    def compute_rates(data: pd.DataFrame) -> Dict[str, float]:
        """Compute rates for a subset of data."""
        n = len(data)
        if n == 0:
            return {}
        
        rates = {'n_trials': n}
        
        # High-level categories
        for col in category_cols:
            # Handle NaN values - treat as False
            rates[col] = data[col].fillna(False).sum() / n
        
        # Low-level flags
        for flag in flag_cols:
            # Handle NaN values - treat as False
            rates[flag] = data[flag].fillna(False).sum() / n
            
        return rates
    
    if group_by and group_by in df_filtered.columns:
        # Compute per-group
        groups = df_filtered.groupby(group_by)
        results = []
        for name, group_df in groups:
            rates = compute_rates(group_df)
            rates[group_by] = name
            results.append(rates)
        stats_df = pd.DataFrame(results)
        if group_by in stats_df.columns:
            stats_df = stats_df.set_index(group_by)
    else:
        # Compute overall
        rates = compute_rates(df_filtered)
        stats_df = pd.DataFrame([rates])
        stats_df.index = ['Overall']
    
    # Split into high-level and low-level
    high_level_cols = ['n_trials'] + category_cols
    low_level_cols = ['n_trials'] + flag_cols
    
    high_level_df = stats_df[[c for c in high_level_cols if c in stats_df.columns]].copy()
    low_level_df = stats_df[[c for c in low_level_cols if c in stats_df.columns]].copy()
    
    # Rename columns for readability
    high_level_df.columns = [
        c.replace('category_', '').replace('_', ' ').title() if c != 'n_trials' else 'N'
        for c in high_level_df.columns
    ]
    low_level_df.columns = [
        c.replace('flag_', '').replace('_', ' ').title() if c != 'n_trials' else 'N'
        for c in low_level_df.columns
    ]
    
    return {
        'high_level': high_level_df,
        'low_level': low_level_df
    }

In [49]:
def analyze_ask_user_failure_modes(
    run_dirs: List[Path],
    group_by: str = 'run_model',
) -> Dict[str, pd.DataFrame]:
    """
    Generate all 6 failure mode tables from ask_user runs.
    
    Args:
        run_dirs: List of paths to run directories with ask_user.csv files
        group_by: Column to group by ('run_name' or 'run_model')
        
    Returns:
        Dict with 6 tables:
            - table1_high_level_all: % of all trials with each high-level failure mode
            - table2_low_level_all: % of all trials with each low-level failure mode
            - table3_high_level_with_question: % of trials WITH questions with each high-level failure
            - table4_low_level_with_question: % of trials WITH questions with each low-level failure
            - table5_high_level_failed: % of FAILED trials with each high-level failure mode
            - table6_low_level_failed: % of FAILED trials with each low-level failure mode
    """
    # Load and prepare data
    df = load_ask_user_data(run_dirs)
    df = add_derived_columns(df)
    
    tables = {}
    
    # Tables 1 & 2: All trials
    print("\n" + "=" * 60)
    print("Tables 1 & 2: ALL TRIALS")
    print("=" * 60)
    stats = compute_failure_mode_stats(df, group_by=group_by)
    tables['table1_high_level_all'] = stats['high_level']
    tables['table2_low_level_all'] = stats['low_level']
    
    # Tables 3 & 4: Trials with questions
    print("\n" + "=" * 60)
    print("Tables 3 & 4: TRIALS WITH QUESTIONS (ask_user_count > 0)")
    print("=" * 60)
    stats = compute_failure_mode_stats(df, group_by=group_by, filter_col='has_question', filter_value=True)
    tables['table3_high_level_with_question'] = stats['high_level']
    tables['table4_low_level_with_question'] = stats['low_level']
    
    # Tables 5 & 6: Failed trials
    print("\n" + "=" * 60)
    print(f"Tables 5 & 6: FAILED TRIALS (coverage_score < {PASS_THRESHOLD})")
    print("=" * 60)
    stats = compute_failure_mode_stats(df, group_by=group_by, filter_col='failed', filter_value=True)
    tables['table5_high_level_failed'] = stats['high_level']
    tables['table6_low_level_failed'] = stats['low_level']
    
    return tables, df

In [50]:
def display_tables(tables: Dict[str, pd.DataFrame]):
    """Display all tables with nice formatting."""
    
    table_titles = {
        'table1_high_level_all': 'Table 1: % of ALL trials with each HIGH-LEVEL failure mode',
        'table2_low_level_all': 'Table 2: % of ALL trials with each LOW-LEVEL failure mode',
        'table3_high_level_with_question': 'Table 3: % of trials WITH QUESTIONS with each HIGH-LEVEL failure mode',
        'table4_low_level_with_question': 'Table 4: % of trials WITH QUESTIONS with each LOW-LEVEL failure mode',
        'table5_high_level_failed': 'Table 5: % of FAILED trials with each HIGH-LEVEL failure mode',
        'table6_low_level_failed': 'Table 6: % of FAILED trials with each LOW-LEVEL failure mode',
    }
    
    for key, title in table_titles.items():
        if key in tables and not tables[key].empty:
            print("\n" + "=" * 80)
            print(title)
            print("=" * 80)
            
            # Format the dataframe for display
            display_df = tables[key].copy()
            
            # Convert N column to int if present
            if 'N' in display_df.columns:
                display_df['N'] = display_df['N'].astype(int)
            
            # Display
            display(display_df)
        else:
            print(f"\n{title}: No data available")

## Usage

Specify the run directories containing `ask_user.csv` files below:

In [51]:
# Define run directories (each must contain an ask_user.csv file)
# You can either:
#   1. Explicitly list run directories
#   2. Auto-discover runs matching a pattern

BASE_DIR = Path("../../experiments/mcpatlas/experiments/mcpatlas/underspec_datasets/intersection_delete_2segments_20260121_161230")

# Option 1: Explicitly list run directories
RUN_DIRS = [
    BASE_DIR / "run_sampled_underspecified_with_user_new_claude-opus-4-5-20251101_20260128_044057",
    BASE_DIR / "run_sampled_underspecified_with_user_new_gemini-3-pro-preview_20260128_155002",
    BASE_DIR / "run_sampled_underspecified_with_user_new_gpt-5.2_20260128_044134",
    BASE_DIR / "run_sampled_underspecified_with_user_new_claude-sonnet-4-5-20250929_20260128_044149",
    BASE_DIR / "run_sampled_underspecified_with_user_new_gemini-3-flash-preview_20260128_044210",
]

# Option 2: Auto-discover runs matching a pattern (e.g., "*_with_user_new_*")
RUN_PATTERN = "*_with_user_new_*"  # Change this pattern as needed
# RUN_DIRS = sorted([
#     d for d in BASE_DIR.glob(f"run{RUN_PATTERN}")
#     if (d / "ask_user.csv").exists()
# ])

print(f"Found {len(RUN_DIRS)} run directories with ask_user.csv:")
for d in RUN_DIRS:
    print(f"  - {d.name}")

Found 5 run directories with ask_user.csv:
  - run_sampled_underspecified_with_user_new_claude-opus-4-5-20251101_20260128_044057
  - run_sampled_underspecified_with_user_new_gemini-3-pro-preview_20260128_155002
  - run_sampled_underspecified_with_user_new_gpt-5.2_20260128_044134
  - run_sampled_underspecified_with_user_new_claude-sonnet-4-5-20250929_20260128_044149
  - run_sampled_underspecified_with_user_new_gemini-3-flash-preview_20260128_044210


In [52]:
# Run the analysis
# group_by options:
#   'run_label' - short readable label for each run (e.g., "gpt-5.2 (new)")
#   'run_name'  - full directory name for each run
#   'run_model' - combines runs with the same model
if RUN_DIRS:
    tables, df = analyze_ask_user_failure_modes(RUN_DIRS, group_by='run_label')
else:
    print("No run directories found. Please update RUN_DIRS above.")

Loaded 300 trials from run_sampled_underspecified_with_user_new_claude-opus-4-5-20251101_20260128_044057
Loaded 300 trials from run_sampled_underspecified_with_user_new_gemini-3-pro-preview_20260128_155002
Loaded 300 trials from run_sampled_underspecified_with_user_new_gpt-5.2_20260128_044134
Loaded 300 trials from run_sampled_underspecified_with_user_new_claude-sonnet-4-5-20250929_20260128_044149
Loaded 300 trials from run_sampled_underspecified_with_user_new_gemini-3-flash-preview_20260128_044210

Total: 1500 trials from 5 runs



Tables 1 & 2: ALL TRIALS

Tables 3 & 4: TRIALS WITH QUESTIONS (ask_user_count > 0)

Tables 5 & 6: FAILED TRIALS (coverage_score < 0.75)


/var/folders/4_/cvd4nrqx09z3hfvp3024hv_r0000gq/T/ipykernel_29290/523182888.py:51: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  rates[flag] = data[flag].fillna(False).sum() / n
/var/folders/4_/cvd4nrqx09z3hfvp3024hv_r0000gq/T/ipykernel_29290/523182888.py:51: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  rates[flag] = data[flag].fillna(False).sum() / n
/var/folders/4_/cvd4nrqx09z3hfvp3024hv_r0000gq/T/ipykernel_29290/523182888.py:51: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version.

In [53]:
# Display all tables
if RUN_DIRS:
    display_tables(tables)


Table 1: % of ALL trials with each HIGH-LEVEL failure mode


,N,Question Quality,Question Targeting,Information Integration,Over-Clarification,Under-Clarification,Timing And Strategy,User Response Misinterpretation
run_label,,,,,,,,
sampled_underspecified_with_user_new_claude-opus-4-5-20251101 (new),300,39.7%,24.7%,4.7%,9.7%,15.7%,1.0%,8.3%
sampled_underspecified_with_user_new_claude-sonnet-4-5-20250929 (new),300,43.0%,25.3%,5.3%,12.3%,13.7%,1.7%,12.3%
sampled_underspecified_with_user_new_gemini-3-flash-preview (new),300,19.7%,15.3%,5.3%,3.3%,13.7%,0.0%,3.7%
sampled_underspecified_with_user_new_gemini-3-pro-preview (new),300,26.3%,18.3%,5.7%,6.0%,13.7%,0.0%,6.0%
sampled_underspecified_with_user_new_gpt-5.2 (new),300,90.7%,44.0%,14.0%,35.0%,19.7%,3.3%,28.0%



Table 2: % of ALL trials with each LOW-LEVEL failure mode


,N,Vague Questions,Compound Questions,Closed Questions,Missed Critical Segments,Wrong Dimension Focus,Inferable Information Asked,Ignoring User Response,Incorrect Information Application,Partial Information Utilization,Redundant Questions,Unnecessary Questions,Excessive Cost,Premature Execution,Insufficient Question Depth,False Confidence,Late Clarification,Upfront Question Dump,Inefficient Question Sequence,Literal Misunderstanding,Context Misapplication,Implicit Information Missed
run_label,,,,,,,,,,,,,,,,,,,,,,
sampled_underspecified_with_user_new_claude-opus-4-5-20251101 (new),300,0.0%,39.3%,0.3%,23.3%,4.7%,1.7%,2.0%,0.0%,2.7%,0.3%,9.3%,0.0%,0.3%,15.0%,2.3%,1.0%,0.0%,0.0%,0.7%,0.7%,7.7%
sampled_underspecified_with_user_new_claude-sonnet-4-5-20250929 (new),300,0.0%,42.7%,0.7%,22.7%,6.3%,2.7%,1.0%,0.3%,4.7%,1.7%,11.0%,0.7%,0.0%,13.0%,1.7%,0.0%,0.3%,1.3%,0.0%,0.3%,12.3%
sampled_underspecified_with_user_new_gemini-3-flash-preview (new),300,0.0%,19.7%,0.0%,14.3%,2.0%,1.0%,3.3%,0.0%,2.3%,0.0%,3.0%,0.3%,0.0%,13.7%,1.7%,0.0%,0.0%,0.0%,0.3%,0.0%,3.3%
sampled_underspecified_with_user_new_gemini-3-pro-preview (new),300,0.3%,26.3%,0.0%,17.7%,3.3%,0.7%,4.0%,0.0%,2.0%,0.3%,5.3%,0.3%,0.0%,13.7%,0.7%,0.0%,0.0%,0.0%,0.0%,0.0%,6.0%
sampled_underspecified_with_user_new_gpt-5.2 (new),300,0.0%,90.7%,1.7%,38.0%,8.3%,5.7%,7.0%,0.0%,8.3%,9.7%,22.7%,9.3%,0.0%,19.3%,1.3%,0.3%,1.7%,1.3%,3.7%,0.0%,26.3%



Table 3: % of trials WITH QUESTIONS with each HIGH-LEVEL failure mode


,N,Question Quality,Question Targeting,Information Integration,Over-Clarification,Under-Clarification,Timing And Strategy,User Response Misinterpretation
run_label,,,,,,,,
sampled_underspecified_with_user_new_claude-opus-4-5-20251101 (new),188,63.3%,39.4%,7.4%,15.4%,25.0%,1.6%,13.3%
sampled_underspecified_with_user_new_claude-sonnet-4-5-20250929 (new),167,77.2%,45.5%,9.6%,22.2%,24.6%,3.0%,22.2%
sampled_underspecified_with_user_new_gemini-3-flash-preview (new),132,44.7%,34.8%,12.1%,7.6%,31.1%,0.0%,8.3%
sampled_underspecified_with_user_new_gemini-3-pro-preview (new),153,51.6%,35.9%,11.1%,11.8%,26.8%,0.0%,11.8%
sampled_underspecified_with_user_new_gpt-5.2 (new),280,97.1%,47.1%,15.0%,37.5%,21.1%,3.6%,30.0%


... (2 outputs left out)


Table 5: % of FAILED trials with each HIGH-LEVEL failure mode


,N,Question Quality,Question Targeting,Information Integration,Over-Clarification,Under-Clarification,Timing And Strategy,User Response Misinterpretation
run_label,,,,,,,,
sampled_underspecified_with_user_new_claude-opus-4-5-20251101 (new),110,28.2%,30.9%,5.5%,13.6%,24.5%,2.7%,11.8%
sampled_underspecified_with_user_new_claude-sonnet-4-5-20250929 (new),137,31.4%,23.4%,2.2%,10.2%,17.5%,0.7%,10.2%
sampled_underspecified_with_user_new_gemini-3-flash-preview (new),133,12.0%,13.5%,9.8%,3.8%,12.8%,0.0%,3.0%
sampled_underspecified_with_user_new_gemini-3-pro-preview (new),122,20.5%,17.2%,11.5%,4.9%,16.4%,0.0%,7.4%
sampled_underspecified_with_user_new_gpt-5.2 (new),145,90.3%,55.9%,20.0%,42.1%,33.1%,4.8%,34.5%



Table 6: % of FAILED trials with each LOW-LEVEL failure mode


,N,Vague Questions,Compound Questions,Closed Questions,Missed Critical Segments,Wrong Dimension Focus,Inferable Information Asked,Ignoring User Response,Incorrect Information Application,Partial Information Utilization,Redundant Questions,Unnecessary Questions,Excessive Cost,Premature Execution,Insufficient Question Depth,False Confidence,Late Clarification,Upfront Question Dump,Inefficient Question Sequence,Literal Misunderstanding,Context Misapplication,Implicit Information Missed
run_label,,,,,,,,,,,,,,,,,,,,,,
sampled_underspecified_with_user_new_claude-opus-4-5-20251101 (new),110,0.0%,28.2%,0.0%,30.9%,8.2%,1.8%,2.7%,0.0%,2.7%,0.0%,13.6%,0.0%,0.9%,23.6%,5.5%,2.7%,0.0%,0.0%,0.9%,0.9%,10.9%
sampled_underspecified_with_user_new_claude-sonnet-4-5-20250929 (new),137,0.0%,31.4%,0.0%,21.2%,5.1%,3.6%,0.7%,0.0%,2.2%,0.0%,10.2%,0.7%,0.0%,16.8%,2.9%,0.0%,0.7%,0.0%,0.0%,0.0%,10.2%
sampled_underspecified_with_user_new_gemini-3-flash-preview (new),133,0.0%,12.0%,0.0%,12.8%,3.8%,0.8%,7.5%,0.0%,3.0%,0.0%,3.0%,0.8%,0.0%,12.8%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,3.0%
sampled_underspecified_with_user_new_gemini-3-pro-preview (new),122,0.8%,20.5%,0.0%,17.2%,7.4%,0.0%,9.8%,0.0%,2.5%,0.8%,3.3%,0.8%,0.0%,16.4%,0.8%,0.0%,0.0%,0.0%,0.0%,0.0%,7.4%
sampled_underspecified_with_user_new_gpt-5.2 (new),145,0.0%,90.3%,2.8%,49.7%,13.8%,6.9%,12.4%,0.0%,10.3%,15.2%,20.0%,19.3%,0.0%,32.4%,2.1%,0.0%,2.1%,2.8%,5.5%,0.0%,33.1%


In [54]:
# Save Table 3: High-level failure modes for trials WITH QUESTIONS
if RUN_DIRS and 'table3_high_level_with_question' in tables:
    output_path = Path("table3_high_level_with_questions.csv")
    tables['table3_high_level_with_question'].to_csv(output_path)
    print(f"Saved: {output_path}")

Saved: table3_high_level_with_questions.csv


## Summary Statistics

In [55]:
def print_summary_stats(df: pd.DataFrame, group_by: str = 'run_label'):
    """Print summary statistics for the loaded data."""
    print("=" * 60)
    print("SUMMARY STATISTICS")
    print("=" * 60)
    
    # Overall stats
    print(f"\nTotal trials: {len(df)}")
    print(f"Trials with questions: {df['has_question'].sum()} ({df['has_question'].mean():.1%})")
    print(f"Trials passed: {df['passed'].sum()} ({df['passed'].mean():.1%})")
    print(f"Trials failed: {df['failed'].sum()} ({df['failed'].mean():.1%})")
    
    # Per-run stats
    print("\n" + "-" * 60)
    print("Per-Run Summary:")
    print("-" * 60)
    
    summary = df.groupby(group_by).agg({
        'task_id': 'count',
        'has_question': ['sum', 'mean'],
        'passed': ['sum', 'mean'],
        'ask_user_count': 'mean',
        'coverage_score': 'mean',
    }).round(3)
    
    summary.columns = ['N', 'Has Q (N)', 'Has Q (%)', 'Passed (N)', 'Passed (%)', 
                       'Avg Questions', 'Avg Coverage']
    display(summary)
    
    # Question count distribution
    print("\n" + "-" * 60)
    print("Question Count Distribution:")
    print("-" * 60)
    q_dist = df.groupby(group_by)['ask_user_count'].value_counts().unstack(fill_value=0)
    display(q_dist)

if RUN_DIRS:
    print_summary_stats(df, group_by='run_label')

SUMMARY STATISTICS

Total trials: 1500
Trials with questions: 920 (61.3%)
Trials passed: 853 (56.9%)
Trials failed: 647 (43.1%)

------------------------------------------------------------
Per-Run Summary:
------------------------------------------------------------


,N,Has Q (N),Has Q (%),Passed (N),Passed (%),Avg Questions,Avg Coverage
run_label,,,,,,,
sampled_underspecified_with_user_new_claude-opus-4-5-20251101 (new),300,188,62.7%,190,63.3%,69.0%,71.0%
sampled_underspecified_with_user_new_claude-sonnet-4-5-20250929 (new),300,167,55.7%,163,54.3%,62.0%,62.6%
sampled_underspecified_with_user_new_gemini-3-flash-preview (new),300,132,44.0%,167,55.7%,46.0%,62.0%
sampled_underspecified_with_user_new_gemini-3-pro-preview (new),300,153,51.0%,178,59.3%,54.0%,67.1%
sampled_underspecified_with_user_new_gpt-5.2 (new),300,280,93.3%,155,51.7%,161.3%,62.5%



------------------------------------------------------------
Question Count Distribution:
------------------------------------------------------------


ask_user_count,0,1,2,3,4,5,6
run_label,,,,,,,
sampled_underspecified_with_user_new_claude-opus-4-5-20251101 (new),112,171,15,2,0,0,0
sampled_underspecified_with_user_new_claude-sonnet-4-5-20250929 (new),133,153,12,1,0,0,1
sampled_underspecified_with_user_new_gemini-3-flash-preview (new),168,126,6,0,0,0,0
sampled_underspecified_with_user_new_gemini-3-pro-preview (new),147,145,7,1,0,0,0
sampled_underspecified_with_user_new_gpt-5.2 (new),20,146,87,31,10,5,1


## Additional Analysis: Failed Trials with Questions

This focuses on trials that both asked questions AND failed, showing what went wrong with the clarification process.

In [56]:
def analyze_failed_with_questions(df: pd.DataFrame, group_by: str = 'run_model'):
    """Analyze trials that asked questions but still failed.
    
    Returns:
        Tuple of (filtered_df, stats_dict) where stats_dict has 'high_level' and 'low_level' DataFrames
    """
    # Filter to trials with questions that failed
    failed_with_q = df[(df['has_question']) & (df['failed'])].copy()
    
    if len(failed_with_q) == 0:
        print("No trials found with questions that also failed.")
        return None, None
    
    print(f"Trials that asked questions but FAILED: {len(failed_with_q)}")
    print(f"(out of {df['has_question'].sum()} trials with questions)")
    print(f"Failure rate among question-askers: {len(failed_with_q) / df['has_question'].sum():.1%}")
    
    # Compute failure mode stats for this subset
    stats = compute_failure_mode_stats(failed_with_q, group_by=group_by)
    
    print("\n" + "=" * 80)
    print("HIGH-LEVEL failure modes in trials that asked questions but failed:")
    print("=" * 80)
    display(stats['high_level'])
    
    print("\n" + "=" * 80)
    print("LOW-LEVEL failure modes in trials that asked questions but failed:")
    print("=" * 80)
    display(stats['low_level'])
    
    return failed_with_q, stats

if RUN_DIRS:
    failed_with_q_df, failed_with_q_stats = analyze_failed_with_questions(df, group_by='run_label')

Trials that asked questions but FAILED: 309
(out of 920 trials with questions)
Failure rate among question-askers: 33.6%

HIGH-LEVEL failure modes in trials that asked questions but failed:


/var/folders/4_/cvd4nrqx09z3hfvp3024hv_r0000gq/T/ipykernel_29290/523182888.py:51: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  rates[flag] = data[flag].fillna(False).sum() / n


,N,Question Quality,Question Targeting,Information Integration,Over-Clarification,Under-Clarification,Timing And Strategy,User Response Misinterpretation
run_label,,,,,,,,
sampled_underspecified_with_user_new_claude-opus-4-5-20251101 (new),50,62.0%,68.0%,12.0%,30.0%,54.0%,6.0%,26.0%
sampled_underspecified_with_user_new_claude-sonnet-4-5-20250929 (new),53,81.1%,60.4%,5.7%,26.4%,45.3%,1.9%,26.4%
sampled_underspecified_with_user_new_gemini-3-flash-preview (new),32,50.0%,56.2%,40.6%,15.6%,53.1%,0.0%,12.5%
sampled_underspecified_with_user_new_gemini-3-pro-preview (new),40,62.5%,52.5%,35.0%,15.0%,50.0%,0.0%,22.5%
sampled_underspecified_with_user_new_gpt-5.2 (new),134,97.8%,60.4%,21.6%,45.5%,35.8%,5.2%,37.3%



LOW-LEVEL failure modes in trials that asked questions but failed:


,N,Vague Questions,Compound Questions,Closed Questions,Missed Critical Segments,Wrong Dimension Focus,Inferable Information Asked,Ignoring User Response,Incorrect Information Application,Partial Information Utilization,Redundant Questions,Unnecessary Questions,Excessive Cost,Premature Execution,Insufficient Question Depth,False Confidence,Late Clarification,Upfront Question Dump,Inefficient Question Sequence,Literal Misunderstanding,Context Misapplication,Implicit Information Missed
run_label,,,,,,,,,,,,,,,,,,,,,,
sampled_underspecified_with_user_new_claude-opus-4-5-20251101 (new),50,0.0%,62.0%,0.0%,68.0%,18.0%,4.0%,6.0%,0.0%,6.0%,0.0%,30.0%,0.0%,2.0%,52.0%,12.0%,6.0%,0.0%,0.0%,2.0%,2.0%,24.0%
sampled_underspecified_with_user_new_claude-sonnet-4-5-20250929 (new),53,0.0%,81.1%,0.0%,54.7%,13.2%,9.4%,1.9%,0.0%,5.7%,0.0%,26.4%,1.9%,0.0%,43.4%,7.5%,0.0%,1.9%,0.0%,0.0%,0.0%,26.4%
sampled_underspecified_with_user_new_gemini-3-flash-preview (new),32,0.0%,50.0%,0.0%,53.1%,15.6%,3.1%,31.2%,0.0%,12.5%,0.0%,12.5%,3.1%,0.0%,53.1%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,12.5%
sampled_underspecified_with_user_new_gemini-3-pro-preview (new),40,2.5%,62.5%,0.0%,52.5%,22.5%,0.0%,30.0%,0.0%,7.5%,2.5%,10.0%,2.5%,0.0%,50.0%,2.5%,0.0%,0.0%,0.0%,0.0%,0.0%,22.5%
sampled_underspecified_with_user_new_gpt-5.2 (new),134,0.0%,97.8%,3.0%,53.7%,14.9%,7.5%,13.4%,0.0%,11.2%,16.4%,21.6%,20.9%,0.0%,35.1%,2.2%,0.0%,2.2%,3.0%,6.0%,0.0%,35.8%


In [57]:
# Save High-level failure modes for trials that asked questions but FAILED
if RUN_DIRS and failed_with_q_stats is not None:
    output_path = Path("high_level_failed_with_questions.csv")
    failed_with_q_stats['high_level'].to_csv(output_path)
    print(f"Saved: {output_path}")

Saved: high_level_failed_with_questions.csv


## Export Tables

Export all tables to CSV or markdown format:

In [58]:
def export_tables(tables: Dict[str, pd.DataFrame], output_dir: Path, format: str = 'csv'):
    """
    Export all tables to files.
    
    Args:
        tables: Dict of table name -> DataFrame
        output_dir: Directory to save files
        format: 'csv' or 'markdown'
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    for name, df in tables.items():
        if df.empty:
            continue
            
        if format == 'csv':
            path = output_dir / f"{name}.csv"
            df.to_csv(path)
            print(f"Saved: {path}")
        elif format == 'markdown':
            path = output_dir / f"{name}.md"
            with open(path, 'w') as f:
                f.write(df.to_markdown())
            print(f"Saved: {path}")

def print_tables_as_markdown(tables: Dict[str, pd.DataFrame]):
    """Print all tables in markdown format for easy copy-paste."""
    table_titles = {
        'table1_high_level_all': 'Table 1: % of ALL trials with each HIGH-LEVEL failure mode',
        'table2_low_level_all': 'Table 2: % of ALL trials with each LOW-LEVEL failure mode',
        'table3_high_level_with_question': 'Table 3: % of trials WITH QUESTIONS with each HIGH-LEVEL failure mode',
        'table4_low_level_with_question': 'Table 4: % of trials WITH QUESTIONS with each LOW-LEVEL failure mode',
        'table5_high_level_failed': 'Table 5: % of FAILED trials with each HIGH-LEVEL failure mode',
        'table6_low_level_failed': 'Table 6: % of FAILED trials with each LOW-LEVEL failure mode',
    }
    
    for key, title in table_titles.items():
        if key in tables and not tables[key].empty:
            print(f"\n### {title}\n")
            print(tables[key].to_markdown())

# Uncomment to export tables:
# export_tables(tables, Path("./ask_user_analysis_output"), format='csv')

# Print as markdown for easy copy-paste:
if RUN_DIRS:
    print_tables_as_markdown(tables)


### Table 1: % of ALL trials with each HIGH-LEVEL failure mode

| run_label                                                             |   N |   Question Quality |   Question Targeting |   Information Integration |   Over-Clarification |   Under-Clarification |   Timing And Strategy |   User Response Misinterpretation |
|:----------------------------------------------------------------------|----:|-------------------:|---------------------:|--------------------------:|---------------------:|----------------------:|----------------------:|----------------------------------:|
| sampled_underspecified_with_user_new_claude-opus-4-5-20251101 (new)   | 300 |           0.396667 |             0.246667 |                 0.0466667 |            0.0966667 |              0.156667 |             0.01      |                         0.0833333 |
| sampled_underspecified_with_user_new_claude-sonnet-4-5-20250929 (new) | 300 |           0.43     |             0.253333 |                 0.0533333 |       